# Assignment 2 - Part A: MNIST Digit Recognition

This notebook completes Part A of Assignment 2:

1. Train the CNNs from Examples 2 and 3 and plot training/validation accuracy and loss.
2. Resize MNIST images from 28x28 to 7x7 using `cv2.resize`.
3. Train one-hidden-layer BP-ANNs with different hidden-layer sizes.
4. Train two-hidden-layer BP-ANNs with different hidden-layer sizes.
5. Compare the number of trainable parameters with classification accuracy.

For faster execution in Google Colab, select **Runtime > Change runtime type > T4 GPU**.

## 1. Setup and experiment configuration

Change the values below if you want shorter trial runs or longer final runs. The test set is reserved for final evaluation; 10% of the original training set is used for validation.

In [ ]:
import os
import random
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.datasets import mnist
from tensorflow.keras.layers import Conv2D, Dense, Dropout, Flatten, Input, MaxPooling2D
from tensorflow.keras.utils import to_categorical

SEED = 7
VALIDATION_SPLIT = 0.10
BATCH_SIZE = 200
CNN_EPOCHS = 10
ANN_EPOCHS = 20
PATIENCE = 3

# Set to True for a quick code check before running the complete experiment.
FAST_MODE = False
if FAST_MODE:
    CNN_EPOCHS = 2
    ANN_EPOCHS = 3

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Load and inspect MNIST

In [ ]:
(x_train_raw, y_train_labels), (x_test_raw, y_test_labels) = mnist.load_data()

if FAST_MODE:
    x_train_raw = x_train_raw[:12000]
    y_train_labels = y_train_labels[:12000]
    x_test_raw = x_test_raw[:2000]
    y_test_labels = y_test_labels[:2000]

NUM_CLASSES = 10
y_train_onehot = to_categorical(y_train_labels, NUM_CLASSES)
y_test_onehot = to_categorical(y_test_labels, NUM_CLASSES)

print("Training images:", x_train_raw.shape)
print("Training labels:", y_train_labels.shape)
print("Test images:", x_test_raw.shape)
print("Test labels:", y_test_labels.shape)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for digit, ax in enumerate(axes.ravel()):
    index = np.flatnonzero(y_train_labels == digit)[0]
    ax.imshow(x_train_raw[index], cmap="gray")
    ax.set_title(f"Label: {digit}")
    ax.axis("off")
plt.suptitle("Original 28x28 MNIST samples")
plt.tight_layout()
plt.show()

## 3. Examples 2 and 3: CNN learning curves

Each CNN is trained with an internal validation split. The two requested figures show `accuracy` against `val_accuracy` and `loss` against `val_loss`.

In [ ]:
x_train_cnn = x_train_raw.astype("float32")[..., np.newaxis] / 255.0
x_test_cnn = x_test_raw.astype("float32")[..., np.newaxis] / 255.0

def build_example_2_cnn():
    model = Sequential([
        Input(shape=(28, 28, 1)),
        Conv2D(32, (5, 5), activation="relu"),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.2),
        Flatten(),
        Dense(128, activation="relu"),
        Dense(NUM_CLASSES, activation="softmax"),
    ], name="example_2_cnn")
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model

def build_example_3_cnn():
    model = Sequential([
        Input(shape=(28, 28, 1)),
        Conv2D(64, (3, 3), activation="relu"),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(32, (3, 3), activation="relu"),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),
        Flatten(),
        Dense(128, activation="relu"),
        Dropout(0.5),
        Dense(NUM_CLASSES, activation="softmax"),
    ], name="example_3_cnn")
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model

def plot_training_history(history, title):
    epochs = range(1, len(history.history["loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history.history["accuracy"], marker="o", label="Training accuracy")
    axes[0].plot(epochs, history.history["val_accuracy"], marker="o", label="Validation accuracy")
    axes[0].set(xlabel="Epoch", ylabel="Accuracy", title=f"{title}: accuracy")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].plot(epochs, history.history["loss"], marker="o", label="Training loss")
    axes[1].plot(epochs, history.history["val_loss"], marker="o", label="Validation loss")
    axes[1].set(xlabel="Epoch", ylabel="Loss", title=f"{title}: loss")
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
cnn_results = []
cnn_histories = {}

for example_name, builder in [("Example 2", build_example_2_cnn), ("Example 3", build_example_3_cnn)]:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    model = builder()
    print(f"\n{example_name}")
    model.summary()

    start = time.perf_counter()
    history = model.fit(
        x_train_cnn,
        y_train_onehot,
        validation_split=VALIDATION_SPLIT,
        epochs=CNN_EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=True,
        verbose=2,
    )
    elapsed = time.perf_counter() - start
    test_loss, test_accuracy = model.evaluate(x_test_cnn, y_test_onehot, verbose=0)

    cnn_histories[example_name] = history
    cnn_results.append({
        "model": example_name,
        "parameters": model.count_params(),
        "best_val_accuracy": max(history.history["val_accuracy"]),
        "test_accuracy": test_accuracy,
        "test_loss": test_loss,
        "training_seconds": elapsed,
    })
    plot_training_history(history, example_name)

cnn_results_df = pd.DataFrame(cnn_results)
cnn_results_df.style.format({
    "best_val_accuracy": "{:.4f}",
    "test_accuracy": "{:.4f}",
    "test_loss": "{:.4f}",
    "training_seconds": "{:.1f}",
})

### Notes for interpreting the CNN graphs

After running the cell, comment on the actual graphs rather than assuming the result in advance. Useful observations include:

- whether training and validation accuracy rise and then level off;
- whether validation loss stops improving while training loss continues to fall;
- the size of the gap between training and validation accuracy;
- whether the deeper Example 3 CNN improves test accuracy enough to justify its complexity.

## 4. Resize MNIST images to 7x7

`cv2.INTER_AREA` is suitable for reducing image size. Each 7x7 image is flattened into 49 values before it is passed to the fully connected ANN.

In [ ]:
def resize_mnist_to_7x7(images):
    return np.stack([
        cv2.resize(image, (7, 7), interpolation=cv2.INTER_AREA)
        for image in images
    ])

x_train_7x7_images = resize_mnist_to_7x7(x_train_raw)
x_test_7x7_images = resize_mnist_to_7x7(x_test_raw)

x_train_7x7 = x_train_7x7_images.reshape(len(x_train_7x7_images), 49).astype("float32") / 255.0
x_test_7x7 = x_test_7x7_images.reshape(len(x_test_7x7_images), 49).astype("float32") / 255.0

print("7x7 training images:", x_train_7x7_images.shape)
print("ANN training inputs:", x_train_7x7.shape)
print("ANN test inputs:", x_test_7x7.shape)

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(15, 4))
for digit in range(10):
    index = np.flatnonzero(y_train_labels == digit)[0]
    axes[0, digit].imshow(x_train_raw[index], cmap="gray")
    axes[0, digit].set_title(str(digit))
    axes[0, digit].axis("off")
    axes[1, digit].imshow(x_train_7x7_images[index], cmap="gray", interpolation="nearest")
    axes[1, digit].axis("off")
axes[0, 0].set_ylabel("28x28")
axes[1, 0].set_ylabel("7x7")
plt.suptitle("Original and resized MNIST samples")
plt.tight_layout()
plt.show()

## 5. BP-ANN experiment helpers

All ANN configurations use the same optimizer, batch size, epoch limit, validation split, and early-stopping rule so that the comparison is reasonably fair.

In [ ]:
def build_ann(hidden_layers):
    model = Sequential(name="ann_" + "_".join(map(str, hidden_layers)))
    model.add(Input(shape=(49,)))
    for units in hidden_layers:
        model.add(Dense(units, activation="relu"))
    model.add(Dense(NUM_CLASSES, activation="softmax"))
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model

def expected_parameter_count(hidden_layers):
    layer_sizes = [49, *hidden_layers, NUM_CLASSES]
    return sum((inputs + 1) * outputs for inputs, outputs in zip(layer_sizes[:-1], layer_sizes[1:]))

def run_ann_experiments(configurations, experiment_name):
    results = []
    histories = {}

    for hidden_layers in configurations:
        hidden_layers = tuple(hidden_layers)
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(SEED)
        model = build_ann(hidden_layers)

        early_stopping = EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            restore_best_weights=True,
        )
        start = time.perf_counter()
        history = model.fit(
            x_train_7x7,
            y_train_onehot,
            validation_split=VALIDATION_SPLIT,
            epochs=ANN_EPOCHS,
            batch_size=BATCH_SIZE,
            shuffle=True,
            callbacks=[early_stopping],
            verbose=2,
        )
        elapsed = time.perf_counter() - start
        test_loss, test_accuracy = model.evaluate(x_test_7x7, y_test_onehot, verbose=0)
        parameter_count = model.count_params()
        assert parameter_count == expected_parameter_count(hidden_layers)

        label = "-".join(map(str, hidden_layers))
        histories[label] = history
        results.append({
            "experiment": experiment_name,
            "hidden_layers": label,
            "parameters": parameter_count,
            "epochs_trained": len(history.history["loss"]),
            "best_val_accuracy": max(history.history["val_accuracy"]),
            "test_accuracy": test_accuracy,
            "test_loss": test_loss,
            "training_seconds": elapsed,
        })

    return pd.DataFrame(results), histories

def format_results_table(dataframe):
    return dataframe.style.format({
        "best_val_accuracy": "{:.4f}",
        "test_accuracy": "{:.4f}",
        "test_loss": "{:.4f}",
        "training_seconds": "{:.1f}",
    }).background_gradient(subset=["test_accuracy"], cmap="Greens")

## 6. One-hidden-layer BP-ANNs

For one hidden layer containing `h` neurons, the parameter count is:

`(49 + 1)h + (h + 1)10 = 60h + 10`.

In [ ]:
ONE_LAYER_CONFIGS = [(16,), (32,), (64,), (128,)]
one_layer_results, one_layer_histories = run_ann_experiments(
    ONE_LAYER_CONFIGS,
    experiment_name="One hidden layer",
)
format_results_table(one_layer_results)

## 7. Two-hidden-layer BP-ANNs

For hidden layers containing `h1` and `h2` neurons, the parameter count is:

`(49 + 1)h1 + (h1 + 1)h2 + (h2 + 1)10`.

In [ ]:
TWO_LAYER_CONFIGS = [(32, 16), (64, 32), (128, 64), (128, 128)]
two_layer_results, two_layer_histories = run_ann_experiments(
    TWO_LAYER_CONFIGS,
    experiment_name="Two hidden layers",
)
format_results_table(two_layer_results)

## 8. Parameter count and classification accuracy

The following table and graph combine all one- and two-hidden-layer experiments.

In [ ]:
all_ann_results = pd.concat([one_layer_results, two_layer_results], ignore_index=True)
all_ann_results = all_ann_results.sort_values("parameters").reset_index(drop=True)
display(format_results_table(all_ann_results))

fig, ax = plt.subplots(figsize=(8, 5))
for experiment, group in all_ann_results.groupby("experiment"):
    ax.scatter(group["parameters"], group["test_accuracy"], s=80, label=experiment)
    for _, row in group.iterrows():
        ax.annotate(
            row["hidden_layers"],
            (row["parameters"], row["test_accuracy"]),
            xytext=(5, 5),
            textcoords="offset points",
        )
ax.set_xlabel("Number of trainable parameters")
ax.set_ylabel("Test accuracy")
ax.set_title("Model size versus MNIST 7x7 test accuracy")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
best_model_row = all_ann_results.loc[all_ann_results["test_accuracy"].idxmax()]
smallest_model_row = all_ann_results.loc[all_ann_results["parameters"].idxmin()]
largest_model_row = all_ann_results.loc[all_ann_results["parameters"].idxmax()]
accuracy_parameter_correlation = all_ann_results["parameters"].corr(all_ann_results["test_accuracy"])

print("Best ANN configuration:")
print(best_model_row[["experiment", "hidden_layers", "parameters", "test_accuracy"]].to_string())
print(f"\nSmallest model test accuracy: {smallest_model_row['test_accuracy']:.4f}")
print(f"Largest model test accuracy:  {largest_model_row['test_accuracy']:.4f}")
print(f"Pearson correlation between parameters and test accuracy: {accuracy_parameter_correlation:.4f}")

### Points to discuss after running the notebook

Use the generated values to support your discussion:

- Identify the best-performing one-layer and two-layer configurations.
- State whether accuracy consistently increases as the parameter count increases.
- Compare the smallest and largest models and decide whether the additional parameters produce a meaningful improvement.
- Look for overfitting: training performance may continue to improve while validation performance stops improving or becomes worse.
- Explain that reducing 28x28 images to 7x7 decreases input information and makes the classification task harder, but greatly reduces the number of ANN parameters.

## 9. Optional: save result tables

Run this cell if you want CSV copies of the tables for preparing the report later.

In [ ]:
cnn_results_df.to_csv("PartA_CNN_results.csv", index=False)
all_ann_results.to_csv("PartA_ANN_7x7_results.csv", index=False)
print("Saved PartA_CNN_results.csv and PartA_ANN_7x7_results.csv")